# ReadyAI CHiRPE Segmented-In, Segmented-Out Contract

This notebook demonstrates the complete requested boundary: preprocessing runs outside ReadyAI, ReadyAI runs CHiRPE on already segmented input, and aggregation runs outside ReadyAI.

ReadyAI should not do transcript preprocessing or transcript-level aggregation at this stage.

## Responsibility Boundary

| Step | Location |
| --- | --- |
| Raw transcript ingestion | Outside ReadyAI |
| Raw transcript segmentation | Outside ReadyAI |
| Segment summarization | Outside ReadyAI |
| CHiRPE model inference on each segment | Inside ReadyAI |
| Transcript-level voting or aggregation | Outside ReadyAI |

Before / full CHiRPE pipeline:

```text
raw transcript
  -> CHiRPE preprocessing inside the application
  -> segmentation by symptom domain
  -> segment summarization
  -> CHiRPE model inference on segments
  -> transcript-level aggregation/voting
  -> final participant-level output
```

Now / ReadyAI boundary for this stage:

```text
Outside ReadyAI:
  raw transcript
  -> segmentation by symptom domain
  -> segment summarization
  -> segmented transcript payload

Inside ReadyAI:
  segmented transcript payload
  -> CHiRPE model inference on each segment summary
  -> segmented output with one prediction/probability set per segment

Outside ReadyAI:
  segmented output
  -> optional transcript-level aggregation/voting
  -> final participant-level output/report
```

## Outside ReadyAI: Raw Transcript Input

The payload below represents a raw transcript before ReadyAI. This step is outside ReadyAI.

ReadyAI should not receive this raw transcript directly at this stage.

In [ ]:
raw_transcript_payload = {
    "participant_id": "demo_001",
    "transcript": [
        {"speaker": "interviewer", "text": "Tell me about whether events have had special meaning for you."},
        {"speaker": "interviewee", "text": "Sometimes I feel like television messages are personally directed at me."},
        {"speaker": "interviewer", "text": "Tell me whether people are talking about you or watching you."},
        {"speaker": "interviewee", "text": "At times I worry strangers are watching me when I am outside."},
        {"speaker": "interviewer", "text": "Tell me about any trouble concentrating or focusing."},
        {"speaker": "interviewee", "text": "I have mild trouble focusing at school, but I do not get lost or confused about where I am."},
    ],
}

raw_transcript_payload

## Outside ReadyAI: Build Segmented Payload

This cell performs preprocessing outside ReadyAI. It converts the raw transcript into the segmented payload that ReadyAI is allowed to receive.

In [ ]:
from chirpe.data.preprocessor import TranscriptPreprocessor

preprocessor = TranscriptPreprocessor(
    segmentation_threshold=0.80,
    use_llm_summarizer=False,
)

processed = preprocessor.process_transcript(
    raw_transcript_payload["transcript"],
    participant_id=raw_transcript_payload["participant_id"],
)

if not processed["segments"]:
    raise ValueError("Preprocessing produced no segments; check the transcript prompts and threshold.")

segmented_payload = {
    "participant_id": processed["participant_id"],
    "segments": [
        {
            "segment_id": f"{processed['participant_id']}_seg_{index + 1:03d}",
            "domain": segment["domain"],
            "summary": segment["summary"],
            "start_idx": segment["start_idx"],
            "end_idx": segment["end_idx"],
        }
        for index, segment in enumerate(processed["segments"])
    ],
}

segmented_payload

## ReadyAI Input Contract

The object below is the boundary handoff into ReadyAI: a participant ID plus an ordered list of segment summaries.

In [ ]:
import json

print(json.dumps(segmented_payload, indent=2))

## Inside ReadyAI: Load CHiRPE Model

This is the start of the ReadyAI-owned step. ReadyAI loads the CHiRPE classifier and uses the `summary` field from each segment as model input.

Change `MODEL_PATH` to test a different checkpoint.

In [ ]:
from pathlib import Path

from chirpe.models.classifier import CHRClassifier

MODEL_PATH_CANDIDATES = [
    Path("outputs/string_onnx_checkpoints/bert/best_model"),
    Path("../outputs/string_onnx_checkpoints/bert/best_model"),
]
MODEL_PATH = next((path for path in MODEL_PATH_CANDIDATES if path.exists()), MODEL_PATH_CANDIDATES[0])

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model checkpoint not found: {MODEL_PATH}. Update MODEL_PATH to a trained CHiRPE best_model directory."
    )

classifier = CHRClassifier(model_name=str(MODEL_PATH))
classifier.load(MODEL_PATH)

print(f"Loaded CHiRPE model from: {MODEL_PATH}")

## Inside ReadyAI: Segment-Level Prediction Only

This function deliberately calls `classifier.predict()` directly on the segment summaries. It does not call `predict_with_segments()` because that method performs transcript-level aggregation.

In [ ]:
LABELS = {0: "Healthy", 1: "CHR-P"}


def predict_segmented_payload(classifier, payload, text_field="summary"):
    """Run CHiRPE on pre-segmented input and return per-segment outputs only."""
    segments = payload.get("segments", [])
    if not segments:
        raise ValueError("Payload must contain a non-empty 'segments' list.")

    texts = []
    for index, segment in enumerate(segments):
        text = segment.get(text_field)
        if not text:
            raise ValueError(f"Segment {index} is missing required text field: {text_field!r}")
        texts.append(text)

    predictions, probabilities = classifier.predict(texts, return_probs=True)

    segment_outputs = []
    for index, (segment, prediction, probability) in enumerate(
        zip(segments, predictions, probabilities)
    ):
        prediction = int(prediction)
        healthy_prob = float(probability[0])
        chrp_prob = float(probability[1])

        segment_outputs.append(
            {
                "segment_id": segment.get("segment_id", f"segment_{index + 1}"),
                "domain": segment.get("domain", "unknown"),
                "summary": segment[text_field],
                "prediction": prediction,
                "label": LABELS[prediction],
                "confidence": float(max(healthy_prob, chrp_prob)),
                "probabilities": {
                    "Healthy": healthy_prob,
                    "CHR-P": chrp_prob,
                },
            }
        )

    return {
        "participant_id": payload.get("participant_id", "unknown"),
        "segments": segment_outputs,
    }

In [ ]:
readyai_output = predict_segmented_payload(classifier, segmented_payload)
readyai_output

## ReadyAI Output Contract

The output remains segmented. There is one model result per input segment.

The ReadyAI response intentionally does not include a final transcript-level `prediction`, majority vote, average vote, or participant-level CHR-P decision.

In [ ]:
import json

print(json.dumps(readyai_output, indent=2))

## Outside ReadyAI: Aggregate Segment Outputs

This cell performs transcript-level post-processing outside ReadyAI. It consumes ReadyAI's segmented output and produces a participant-level decision.

Do not run this inside the ReadyAI model-serving step.

In [ ]:
def outside_readyai_majority_vote(segment_level_output):
    """Example post-processing that remains outside ReadyAI."""
    votes = [segment["prediction"] for segment in segment_level_output["segments"]]
    chrp_votes = sum(vote == 1 for vote in votes)
    healthy_votes = sum(vote == 0 for vote in votes)
    final_prediction = 1 if chrp_votes > healthy_votes else 0
    return {
        "participant_id": segment_level_output["participant_id"],
        "prediction": final_prediction,
        "label": LABELS[final_prediction],
        "chrp_votes": chrp_votes,
        "healthy_votes": healthy_votes,
    }

aggregated_output = outside_readyai_majority_vote(readyai_output)
aggregated_output

## Optional: Triton/Fused ONNX Equivalent

If ReadyAI calls Triton instead of the Python classifier, the same boundary applies. ReadyAI sends one segment summary as a string input and receives model outputs for that segment.

Expected Triton request shape for one segment:

```json
{
  "inputs": [
    {
      "name": "text",
      "shape": [1],
      "datatype": "BYTES",
      "data": ["segment summary text"]
    }
  ],
  "outputs": [
    {"name": "probabilities"},
    {"name": "label"}
  ]
}
```

Current fused models are under `outputs/string_onnx_fused/`, for example `chirpe_bert_string`.

## Summary

What is inside ReadyAI:

- Load CHiRPE model.
- Accept already segmented transcript payload.
- Run CHiRPE on each segment summary.
- Return per-segment predictions/probabilities.

What stays outside ReadyAI at this stage:

- Convert raw transcript into segmented transcript.
- Summarize raw transcript segments.
- Aggregate segment outputs into a transcript-level decision.
- Generate final reports or clinician-facing post-processing.